# Error Analysis — Extraction Method Comparison

Find common patterns among errors: first by **score** (correctness, completeness), then by **variance** (method disagreement, document inconsistency).

Data: `extraction_method_comparison.json` — 4 methods × documents × columns.

In [1]:
import json
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

# Run from repo root or experiment-analysis/
DATA_PATH = Path('experiment-analysis/extraction_method_comparison.json')
if not DATA_PATH.exists():
    DATA_PATH = Path('extraction_method_comparison.json')

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    rows = json.load(f)
df = pd.DataFrame(rows)

# Errors = correctness < 1 OR completeness < 1
df['is_error'] = (df['correctness'] < 1) | (df['completeness'] < 1)
errors = df[df['is_error']].copy()

print(f"Total rows: {len(df)}")
print(f"Error rows: {len(errors)} ({100*len(errors)/len(df):.1f}%)")

Total rows: 5328
Error rows: 1149 (21.6%)


## Part 1: Common Patterns by Score

Group errors by `(correctness, completeness)` and break down by eval_category, method, and column group.

In [2]:
# Score pattern distribution
score_counts = errors.groupby(['correctness', 'completeness']).size().reset_index(name='count')
score_counts = score_counts.sort_values('count', ascending=False)
score_counts['pct'] = (score_counts['count'] / len(errors) * 100).round(1)

# Human-readable labels for score patterns
def score_label(c, k):
    if c == 0 and k == 0: return 'Both wrong (0, 0)'
    if c == 1 and k == 0.5: return 'Correct but incomplete (1, 0.5)'
    if c == 0.5 and k == 0.5: return 'Partial both (0.5, 0.5)'
    if c == 0 and k == 1: return 'Wrong but "complete" (0, 1)'
    if c == 0.5 and k == 1: return 'Partial correct, full completeness (0.5, 1)'
    return f'({c}, {k})'

score_counts['label'] = score_counts.apply(lambda r: score_label(r['correctness'], r['completeness']), axis=1)

print("=== ERROR COUNT BY SCORE PATTERN ===")
display(score_counts[['label', 'count', 'pct']])

=== ERROR COUNT BY SCORE PATTERN ===


,label,count,pct
0,"Both wrong (0, 0)",722,62.8
7,"Correct but incomplete (1, 0.5)",212,18.5
4,"Partial both (0.5, 0.5)",80,7.0
2,"Wrong but ""complete"" (0, 1)",69,6.0
5,"Partial correct, full completeness (0.5, 1)",55,4.8
3,"(0.5, 0.0)",6,0.5
1,"(0.0, 0.5)",3,0.3
6,"(1.0, 0.0)",2,0.2


In [3]:
# Within each score pattern: breakdown by eval_category
by_score_cat = errors.groupby(['correctness', 'completeness', 'eval_category']).size().reset_index(name='count')
by_score_cat = by_score_cat.sort_values(['correctness', 'completeness', 'count'], ascending=[True, True, False])

print("=== BY SCORE × EVAL_CATEGORY ===")
for (c, k), g in by_score_cat.groupby(['correctness', 'completeness']):
    lbl = score_label(c, k)
    print(f"\n{lbl}:")
    for _, row in g.iterrows():
        print(f"  {row['eval_category']}: {row['count']}")

=== BY SCORE × EVAL_CATEGORY ===

Both wrong (0, 0):
  numeric_tolerance: 555
  structured_text: 84
  exact_match: 83

(0.0, 0.5):
  numeric_tolerance: 3

Wrong but "complete" (0, 1):
  numeric_tolerance: 69

(0.5, 0.0):
  numeric_tolerance: 3
  structured_text: 3

Partial both (0.5, 0.5):
  structured_text: 53
  numeric_tolerance: 27

Partial correct, full completeness (0.5, 1):
  numeric_tolerance: 29
  structured_text: 26

(1.0, 0.0):
  structured_text: 2

Correct but incomplete (1, 0.5):
  numeric_tolerance: 185
  structured_text: 27


In [4]:
# Within each score pattern: breakdown by method
by_score_method = errors.groupby(['correctness', 'completeness', 'method']).size().reset_index(name='count')
by_score_method = by_score_method.sort_values(['correctness', 'completeness', 'count'], ascending=[True, True, False])

print("=== BY SCORE × METHOD ===")
for (c, k), g in by_score_method.groupby(['correctness', 'completeness']):
    lbl = score_label(c, k)
    print(f"\n{lbl}:")
    for _, row in g.iterrows():
        print(f"  {row['method']}: {row['count']}")

=== BY SCORE × METHOD ===

Both wrong (0, 0):
  Gemini with Parsed Document as text: 261
  Gemini-2.5-flash-native: 199
  GPT-4.1 with Parsed Document as text: 170
  Reconciliation: 92

(0.0, 0.5):
  GPT-4.1 with Parsed Document as text: 3

Wrong but "complete" (0, 1):
  GPT-4.1 with Parsed Document as text: 23
  Reconciliation: 21
  Gemini-2.5-flash-native: 18
  Gemini with Parsed Document as text: 7

(0.5, 0.0):
  Gemini-2.5-flash-native: 3
  GPT-4.1 with Parsed Document as text: 2
  Reconciliation: 1

Partial both (0.5, 0.5):
  Gemini with Parsed Document as text: 32
  Gemini-2.5-flash-native: 23
  GPT-4.1 with Parsed Document as text: 18
  Reconciliation: 7

Partial correct, full completeness (0.5, 1):
  GPT-4.1 with Parsed Document as text: 31
  Reconciliation: 10
  Gemini with Parsed Document as text: 8
  Gemini-2.5-flash-native: 6

(1.0, 0.0):
  GPT-4.1 with Parsed Document as text: 2

Correct but incomplete (1, 0.5):
  Gemini-2.5-flash-native: 82
  Gemini with Parsed Document a

In [5]:
# Top column groups (group) for each major error pattern
top_patterns = [(0, 0), (1, 0.5), (0.5, 0.5), (0, 1), (0.5, 1)]

for c, k in top_patterns:
    sub = errors[(errors['correctness'] == c) & (errors['completeness'] == k)]
    if sub.empty:
        continue
    grp_counts = sub.groupby('group').size().sort_values(ascending=False).head(10)
    lbl = score_label(c, k)
    print(f"\n=== Top column groups for {lbl} (n={len(sub)}) ===")
    print(grp_counts.to_string())


=== Top column groups for Both wrong (0, 0) (n=722) ===
group
OS Rate (%)                             75
Adverse Events - N (%)                  72
Region - N (%)                          67
Gleason score - N (%)                   44
Reporting by prognostic groups - Y/N    43
Metastases - N (%)                      36
COE_RCT_IND_OVERALL_RJ                  36
Median OS (mo)                          36
Race - N (%)                            31
PFS Rate (%)                            27

=== Top column groups for Correct but incomplete (1, 0.5) (n=212) ===
group
Race - N (%)                        22
OS Rate (%)                         22
Gleason score - N (%)               17
No. of Deaths - N                   16
Volume of disease - N (%)           15
Metastases - N (%)                  15
Median PFS (mo)                     13
Docetaxel administration - N (%)    12
PFS Rate (%)                        10
Treatment Arm 1 Regimen              9

=== Top column groups for Partial both 

In [6]:
# Sample examples: GT vs Predicted for each major score pattern
def sample_for_pattern(c, k, n=3):
    sub = errors[(errors['correctness'] == c) & (errors['completeness'] == k)]
    if sub.empty:
        return None
    samples = sub.sample(min(n, len(sub)), random_state=42)
    return samples[['document', 'method', 'column_name', 'eval_category', 'ground_truth', 'predicted']]

print("=== SAMPLE EXAMPLES BY SCORE PATTERN ===")
for c, k in top_patterns:
    lbl = score_label(c, k)
    s = sample_for_pattern(c, k)
    if s is not None:
        print(f"\n--- {lbl} ---")
        display(s)

=== SAMPLE EXAMPLES BY SCORE PATTERN ===

--- Both wrong (0, 0) ---


,document,method,column_name,eval_category,ground_truth,predicted
2877,NCT01809691_Aggarwal_SWOG1216_JCO'22,Reconciliation,Reporting by prognostic groups - Y/N | High vo...,exact_match,No (extensive vs minimal disease) is reported ...,Y
4030,NCT02446405_Sweeney_ENZAMET_Lancet Onc'23,Gemini with Parsed Document as text,Metastases - N (%) | Nodal | Treatment,numeric_tolerance,,not found
4276,NCT02799602_Hussain_ARASENS_JCO'23,Gemini-2.5-flash-native,Median On-Treatment Duration (mo) | Control,numeric_tolerance,,not found



--- Correct but incomplete (1, 0.5) ---


,document,method,column_name,eval_category,ground_truth,predicted
540,NCT00268476_Attard_STAMPEDE_Lancet'23,Gemini-2.5-flash-native,Median Age (years) | Control,numeric_tolerance,Abiraterone trial: 67 Abiraterone + Enzalutami...,68
4051,NCT02446405_Sweeney_ENZAMET_Lancet Onc'23,Gemini with Parsed Document as text,PFS Rate (%) | Overall | Treatment,numeric_tolerance,PSA Progression Free Survival: 54% at 5 years....,54% (95% 49–58)
3578,NCT01957436_Fizazi_PEACE1_Lancet'22,Gemini with Parsed Document as text,Volume of disease - N (%) | High | Treatment,numeric_tolerance,Overall population: 331 (57%) ADT + docetaxel ...,224 (63%)



--- Partial both (0.5, 0.5) ---


,document,method,column_name,eval_category,ground_truth,predicted
1723,NCT00309985_Kriayako_CHAARTED_JCO'18,Gemini-2.5-flash-native,Secondary Endpoint(s),structured_text,Time to CRPC; time to clinical progression; PS...,"time to development of CRPC, defined either se..."
128,NCT00104715_Gravis_GETUG_EU'15,Gemini-2.5-flash-native,Treatment Class,structured_text,Chemotherapy (taxane),Hormone Therapy and Chemotherapy
1230,NCT00268476_James_STAMPEDE_IJC'22,Reconciliation,Treatment Arm(s),structured_text,SOC + AAP (standard-of-care + abiraterone acet...,Not reported



--- Wrong but "complete" (0, 1) ---


,document,method,column_name,eval_category,ground_truth,predicted
3830,NCT02446405_Sweeney_ENZAMET_Lancet Onc'23,Gemini-2.5-flash-native,Region - N (%) | North America | Control,numeric_tolerance,22 (4%),129 (23%)
2042,NCT00309985_Kriayako_CHAARTED_JCO'18,GPT-4.1 with Parsed Document as text,Median PFS (mo) | High volume | Treatment,numeric_tolerance,27.3 months,14.9 (12.4 to 17.2)
4311,NCT02799602_Hussain_ARASENS_JCO'23,Gemini-2.5-flash-native,Metastases - N (%) | Bone | Treatment,numeric_tolerance,517(79.4%),"1,038 (79.5%)"



--- Partial correct, full completeness (0.5, 1) ---


,document,method,column_name,eval_category,ground_truth,predicted
3711,NCT01957436_Fizazi_PEACE1_Lancet'22,GPT-4.1 with Parsed Document as text,Treatment Arm 1 Regimen,structured_text,Abiraterone 1000 mg PO daily + prednisone 5 mg...,Standard of care (androgen deprivation therapy...
1595,NCT00268476_James_STAMPEDE_IJC'22,GPT-4.1 with Parsed Document as text,Treatment Arm(s),structured_text,SOC + AAP (standard-of-care + abiraterone acet...,Standard-of-care (SOC-alone); Standard-of-care...
3714,NCT01957436_Fizazi_PEACE1_Lancet'22,GPT-4.1 with Parsed Document as text,Treatment Arm(s),structured_text,"SOC + abiraterone, SOC + radiotherapy + abirat...",standard of care (androgen deprivation therapy...


## Part 2: Variance Analysis

**A. Method disagreement**: For each (document, column), compute variance of correctness across the 4 methods. High variance = methods disagree.

**B. Document inconsistency**: For each (group, method), compute variance of correctness across documents. High variance = column type performs inconsistently across trials.

In [7]:
# A. Variance across methods for each (document, column_name)
doc_col_agg = df.groupby(['document', 'column_name']).agg(
    corr_mean=('correctness', 'mean'),
    corr_std=('correctness', 'std'),
    comp_mean=('completeness', 'mean'),
    comp_std=('completeness', 'std'),
    n_methods=('method', 'count'),
).reset_index()
doc_col_agg['corr_std'] = doc_col_agg['corr_std'].fillna(0)
doc_col_agg['comp_std'] = doc_col_agg['comp_std'].fillna(0)

# High variance = at least 2 methods, std > 0.3
high_var_doc_col = doc_col_agg[
    (doc_col_agg['n_methods'] >= 2) & 
    ((doc_col_agg['corr_std'] > 0.3) | (doc_col_agg['comp_std'] > 0.3))
].sort_values('corr_std', ascending=False)

print("=== HIGH VARIANCE (document, column) — methods disagree ===")
print(f"Rows where methods disagree (std > 0.3): {len(high_var_doc_col)}")
display(high_var_doc_col.head(25))

=== HIGH VARIANCE (document, column) — methods disagree ===
Rows where methods disagree (std > 0.3): 388


,document,column_name,corr_mean,corr_std,comp_mean,comp_std,n_methods
158,NCT00268476_Attard_STAMPEDE_Lancet'23,Median OS (mo) | Low volume | Treatment,0.5,0.57735,0.375,0.478714,4
942,NCT02446405_Sweeney_ENZAMET_Lancet Onc'23,Adverse Events - N (%) | Treatment-related Gra...,0.5,0.57735,0.500,0.577350,4
943,NCT02446405_Sweeney_ENZAMET_Lancet Onc'23,Adverse Events - N (%) | Treatment-related Gra...,0.5,0.57735,0.500,0.577350,4
893,NCT01957436_Fizazi_PEACE1_Lancet'22,Quality of Life reported,0.5,0.57735,0.500,0.577350,4
931,NCT01957436_Fizazi_PEACE1_Lancet'22,Treatment Class,0.5,0.57735,0.500,0.577350,4
1103,NCT02799602_Hussain_ARASENS_JCO'23,Median OS (mo) | Overall | Treatment,0.5,0.57735,0.500,0.577350,4
488,NCT00309985_Kriayako_CHAARTED_JCO'18,PubMed ID,0.5,0.57735,0.500,0.577350,4
382,NCT00268476_James_STAMPEDE_IJC'22,Region - N (%) | South America | Control,0.5,0.57735,0.500,0.577350,4
1114,NCT02799602_Hussain_ARASENS_JCO'23,Metastases - N (%) | Liver | Control,0.5,0.57735,0.500,0.577350,4
1209,NCT02799602_Smith_ARASENS_NEJM'22,Add-on Treatment,0.5,0.57735,0.500,0.577350,4


In [8]:
# B. Variance across documents for each (group, method)
group_method_agg = df.groupby(['group', 'method']).agg(
    corr_mean=('correctness', 'mean'),
    corr_std=('correctness', 'std'),
    comp_mean=('completeness', 'mean'),
    comp_std=('completeness', 'std'),
    n_docs=('document', 'nunique'),
    n_rows=('correctness', 'count'),
).reset_index()
group_method_agg['corr_std'] = group_method_agg['corr_std'].fillna(0)

# High variance = inconsistent across documents
high_var_group = group_method_agg[
    (group_method_agg['n_docs'] >= 3) & 
    (group_method_agg['corr_std'] > 0.35)
].sort_values('corr_std', ascending=False)

print("=== HIGH VARIANCE (group, method) — inconsistent across documents ===")
print(f"Column types with high doc-level variance (std > 0.35): {len(high_var_group)}")
display(high_var_group.head(25))

=== HIGH VARIANCE (group, method) — inconsistent across documents ===
Column types with high doc-level variance (std > 0.35): 60


,group,method,corr_mean,corr_std,comp_mean,comp_std,n_docs,n_rows
102,Quality of Life reported,GPT-4.1 with Parsed Document as text,0.500000,0.527046,0.500000,0.527046,10,10
103,Quality of Life reported,Gemini with Parsed Document as text,0.500000,0.527046,0.500000,0.527046,10,10
104,Quality of Life reported,Gemini-2.5-flash-native,0.500000,0.527046,0.500000,0.527046,10,10
83,PFS Rate (%),Gemini with Parsed Document as text,0.500000,0.512989,0.400000,0.447214,10,20
100,Quality of Life Scale,Gemini-2.5-flash-native,0.450000,0.497214,0.450000,0.497214,10,10
7,Adverse Events - N (%),Gemini with Parsed Document as text,0.433333,0.491165,0.516667,0.495460,10,60
8,Adverse Events - N (%),Gemini-2.5-flash-native,0.633333,0.485961,0.600000,0.476570,10,60
115,Reporting by prognostic groups - Y/N,Gemini with Parsed Document as text,0.650000,0.483046,0.650000,0.483046,10,40
98,Quality of Life Scale,GPT-4.1 with Parsed Document as text,0.700000,0.483046,0.700000,0.483046,10,10
99,Quality of Life Scale,Gemini with Parsed Document as text,0.300000,0.483046,0.300000,0.483046,10,10


In [9]:
# Cross-reference: high-variance (doc, col) — show method breakdown
if len(high_var_doc_col) > 0:
    top = high_var_doc_col.head(5)
    for _, r in top.iterrows():
        doc, col = r['document'], r['column_name']
        sub = df[(df['document'] == doc) & (df['column_name'] == col)]
        print(f"\n--- {doc[:50]} | {col[:45]} ---")
        print(sub[['method', 'correctness', 'completeness', 'ground_truth', 'predicted']].to_string())


--- NCT00268476_Attard_STAMPEDE_Lancet'23 | Median OS (mo) | Low volume | Treatment ---
                                    method  correctness  completeness                                                 ground_truth    predicted
563                Gemini-2.5-flash-native          0.0           0.0  Abiraterone trial: 100 Abiraterone + Enzalutamide trial: 80           97
787                         Reconciliation          1.0           0.5  Abiraterone trial: 100 Abiraterone + Enzalutamide trial: 80   100 months
828    Gemini with Parsed Document as text          1.0           1.0  Abiraterone trial: 100 Abiraterone + Enzalutamide trial: 80  100 (95-NR)
1025  GPT-4.1 with Parsed Document as text          0.0           0.0  Abiraterone trial: 100 Abiraterone + Enzalutamide trial: 80   97 (92-NR)

--- NCT02446405_Sweeney_ENZAMET_Lancet Onc'23 | Adverse Events - N (%) | Treatment-related Gr ---
                                    method  correctness  completeness ground_truth     predi

## Summary

- **Score patterns**: (0,0) dominates (both wrong), then (1, 0.5) (correct but incomplete). Most (0,0) are numeric_tolerance; Gemini Parsed has most (0,0) errors.
- **Method disagreement**: High variance (doc, col) = different methods give different scores for same extraction — useful for reconciliation analysis.
- **Document inconsistency**: High variance (group, method) = certain column types (e.g. N (%), OS Rate) are harder on some documents.